# 1. Imports y carga de datos

In [13]:
import pandas as pd 
from pysentimiento import create_analyzer

df = pd.read_csv('../data/processed/reviews_with_text.csv')

print(f'shape inicial : {df.shape}')
print(df['idioma_detectado'].value_counts())

shape inicial : (3537, 16)
idioma_detectado
es             3423
pt               50
en               36
it               11
ca                6
ro                3
cy                2
so                2
fr                1
vi                1
desconocido       1
id                1
Name: count, dtype: int64


# 2. Filtrar solo reseñas en español 

In [14]:
df_es = df[df['idioma_detectado'] == 'es'].copy()
df_otros = df[df['idioma_detectado'] != 'es'].copy()

print(f'Español (van a análisis de sentimiento): {len(df_es)}')
print(f'Otros idiomas (excluidas, documentadas aparte): {len(df_otros)}')

Español (van a análisis de sentimiento): 3423
Otros idiomas (excluidas, documentadas aparte): 114


# 3. Cargar el analizador

In [15]:
analyzer = create_analyzer(task="sentiment", lang='es')

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1768.48it/s]


# 4. Función de análisis

In [16]:
def analizar_sentimiento(texto):
    resultado = analyzer.predict(texto)
    return pd.Series({
        'sentimiento': resultado.output,
        'prob_positivo': resultado.probas['POS'],
        'prob_negativo': resultado.probas['NEG'],
        'prob_neutral': resultado.probas['NEU']
    })
    
df_es[['sentimiento','prob_positivo','prob_negativo','prob_neutral']] = df_es['review_texto'].apply(analizar_sentimiento)

# 5. Revisión general

In [17]:
print('Distribución global de sentimiento: ')
print(df_es['sentimiento'].value_counts(normalize=True) * 100)

print('\nSentimiento por restaurante')
display(df_es.groupby('restaurante')['sentimiento'].value_counts(normalize=True).unstack() * 100)

Distribución global de sentimiento: 
sentimiento
POS    68.010517
NEG    17.791411
NEU    14.198072
Name: proportion, dtype: float64

Sentimiento por restaurante


sentimiento,NEG,NEU,POS
restaurante,,,
Casa Res | Steak House (Mall del Sol),16.488223,18.201285,65.310493
Don Parrilla Steak House - Urdesa,21.641791,11.940299,66.417910
La Casa del Tomahawk,30.161290,7.419355,62.419355
La Parrilla Del Ñato - Urdesa,14.871795,10.256410,74.871795
MoroGrill - C.C. Las Terrazas,10.752688,14.695341,74.551971
Parrillada Punta Del Este,9.424084,21.989529,68.586387
Parrillada Restaurant El Dorado Sauces 3,10.182768,28.981723,60.835509
Rukito Grill&Drink - Alborada,17.831074,10.948905,71.220021


# 6. Validación manual

In [18]:
muestra = df_es.sample(50, random_state=42)[['review_texto','sentimiento','rating']]
print(muestra.to_string())

# 7.Exportar

In [19]:
df_es.to_csv('../data/processed/reviews_with_sentimiento.csv',index=False)
df_otros.to_csv('../data/processed/reviews_excluded_language.csv',index=False)

print('Archivos Guardados correctamente. ')

Archivos Guardados correctamente. 
